In [3]:
from pathlib import Path
import pandas as pd
import logging
from datetime import datetime

root = Path('C:\\Users\\GONCA\\Desktop\\Iscte\\MCD\\Theses\\Dataset')
out_dir = root / 'merged_imp_csv'
out_dir.mkdir(exist_ok=True, parents=True)

# Setup logging
log_path = out_dir / f'merge_errors_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    filename=log_path,
    level=logging.ERROR,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

csv_paths = list(root.glob('*/imp_csv/*.csv'))

if not csv_paths:
    print("No CSV files found under dataset/*/imp_csv/")
else:
    groups = {}
    for p in csv_paths:
        groups.setdefault(p.name, []).append(p)

    ts_candidates = {'timestamp', 'time', 'ts', 'date', 'datetime', 'date_time'}

    def find_ts_col(df):
        for c in df.columns:
            if c.strip().lower() in ts_candidates:
                return c
        return None

    for filename, paths in groups.items():

        dfs = []
        has_pre = False
        has_in = False
        has_post = False

        for p in paths:
            try:
                df = pd.read_csv(p)

                # Identify segment type from folder structure
                parent_name = p.parent.parent.name.lower()

                if "pre" in parent_name:
                    has_pre = True
                elif "in" in parent_name:
                    has_in = True
                elif "post" in parent_name or "pst" in parent_name:
                    has_post = True

                df['_source_folder'] = parent_name
                dfs.append(df)

            except Exception as e:
                msg = f"[{filename}] Failed to read '{p}': {e}"
                print(f"ERROR | {msg}")
                logger.error(msg)

        if not dfs:
            msg = f"[{filename}] No valid files to merge."
            print(f"SKIPPED | {msg}")
            logger.error(msg)
            continue

        try:
            merged = pd.concat(dfs, ignore_index=True, sort=False)

            ts_col = find_ts_col(merged)
            if ts_col:
                merged[ts_col] = pd.to_datetime(merged[ts_col], errors='coerce')
                merged = merged.sort_values(by=ts_col)

            merged = merged.drop_duplicates().reset_index(drop=True)

            # ── Add segment availability flags ─────────────────────
            merged["has_pre"] = int(has_pre)
            merged["has_in"] = int(has_in)
            merged["has_post"] = int(has_post)

            out_path = out_dir / filename
            merged.to_csv(out_path, index=False)

            print(
                f"OK | Wrote '{out_path}' ({len(merged)} rows) "
                f"| pre={has_pre} in={has_in} post={has_post}"
            )

        except Exception as e:
            msg = f"[{filename}] Merge/write failed: {e}"
            print(f"ERROR | {msg}")
            logger.error(msg)

    print(f"\nDone. Error log: '{log_path}' (empty = no errors)")

OK | Wrote 'C:\Users\GONCA\Desktop\Iscte\MCD\Theses\Dataset\merged_imp_csv\0001b3b2f18c01c62ed9b2a87de7b4e33e7836f786f7904471d8866978405c1b.csv' (43946 rows) | pre=True in=True post=True
OK | Wrote 'C:\Users\GONCA\Desktop\Iscte\MCD\Theses\Dataset\merged_imp_csv\0004150214d14a2b2e6f7075531e661cf465b27ec4d0d53573866df2c97b8313.csv' (34178 rows) | pre=True in=True post=True
OK | Wrote 'C:\Users\GONCA\Desktop\Iscte\MCD\Theses\Dataset\merged_imp_csv\000721f0fc6ccf02ae24b67393979513171f2abc119af07a7abed20ca303b665.csv' (43946 rows) | pre=True in=True post=True
OK | Wrote 'C:\Users\GONCA\Desktop\Iscte\MCD\Theses\Dataset\merged_imp_csv\0009b156f2a1213a137c150f4787150384938eac15cc2044a10c797d98667a91.csv' (39266 rows) | pre=True in=True post=True
OK | Wrote 'C:\Users\GONCA\Desktop\Iscte\MCD\Theses\Dataset\merged_imp_csv\000bf84faacf921b55bd4ec4aecda599754e9017e150091d330afc038f0fab11.csv' (19826 rows) | pre=False in=True post=True
OK | Wrote 'C:\Users\GONCA\Desktop\Iscte\MCD\Theses\Dataset\merg